In [12]:
import numpy as np
import pandas as pd

def safe_sharpe_ratio(r):
    """Compute Sharpe ratio with sample std; return np.nan if undefined."""
    sd = np.std(r, ddof=0)
    if sd <= 0 or not np.isfinite(sd):
        return np.nan
    return np.mean(r) / sd


def calculate_hac_variance(r1, r2):
    """
    Newey-West style HAC estimate of covariance of sample moments.

    Returns covariance matrix for sqrt(T)-scaled moments divided by T,
    so it is directly compatible with the delta-method standard error.
    """
    T = len(r1)
    W = np.column_stack((r1, r2, r1**2, r2**2))
    W_centered = W - np.mean(W, axis=0)

    maxlags = int(np.ceil(T ** 0.25))
    Omega = np.dot(W_centered.T, W_centered) / T

    for lag in range(1, maxlags + 1):
        weight = 1.0 - (lag / (maxlags + 1.0))
        Gamma_lag = np.dot(W_centered[lag:].T, W_centered[:-lag]) / T
        Omega += weight * (Gamma_lag + Gamma_lag.T)

    return Omega / T


def calculate_se_from_cov(cov_matrix, r1, r2):
    """Apply delta method gradient for Sharpe ratio difference."""
    mu1, mu2 = np.mean(r1), np.mean(r2)
    gamma1, gamma2 = np.mean(r1**2), np.mean(r2**2)

    var1 = gamma1 - mu1**2
    var2 = gamma2 - mu2**2
    if var1 <= 0 or var2 <= 0:
        return np.nan

    grad = np.zeros(4)
    grad[0] = gamma1 / (var1 ** 1.5)
    grad[1] = -gamma2 / (var2 ** 1.5)
    grad[2] = -0.5 * mu1 / (var1 ** 1.5)
    grad[3] = 0.5 * mu2 / (var2 ** 1.5)

    se2 = np.dot(grad.T, np.dot(cov_matrix, grad))
    if se2 <= 0 or not np.isfinite(se2):
        return np.nan
    return np.sqrt(se2)


def ledoit_wolf_bootstrap(r1, r2, B=4999, block_size=None, random_state=42):
    """
    Ledoit-Wolf (2008), Section 3.2.2 time-series bootstrap test for
    Sharpe ratio difference.
    """
    r1 = np.asarray(r1, dtype=float)
    r2 = np.asarray(r2, dtype=float)

    if len(r1) != len(r2):
        raise ValueError("Return sequences must have the same length.")

    T = len(r1)
    if T < 5:
        raise ValueError("Need at least 10 observations for a meaningful bootstrap test.")

    if block_size is None:
        block_size = max(1, int(np.ceil(T ** (1.0 / 3.0))))
    if block_size > T:
        block_size = T

    l = int(np.floor(T / block_size))
    if l < 1:
        raise ValueError("Block configuration is invalid. Reduce block_size.")

    # 1. Sample statistics
    sr1 = safe_sharpe_ratio(r1)
    sr2 = safe_sharpe_ratio(r2)
    if not np.isfinite(sr1) or not np.isfinite(sr2):
        raise ValueError("Sharpe ratio undefined: one sequence has zero variance.")
    diff_orig = sr1 - sr2

    cov_orig = calculate_hac_variance(r1, r2)
    se_orig = calculate_se_from_cov(cov_orig, r1, r2)
    if not np.isfinite(se_orig) or se_orig <= 0:
        raise ValueError("Original standard error is invalid.")

    d_orig = abs(diff_orig / se_orig)

    # Circular bootstrap setup
    rng = np.random.default_rng(random_state)
    r1_circ = np.concatenate((r1, r1))
    r2_circ = np.concatenate((r2, r2))

    d_stars = np.full(B, np.nan)

    # 2. Bootstrap iterations
    for i in range(B):
        n_blocks = int(np.ceil(T / block_size))
        start_indices = rng.integers(0, T, size=n_blocks)
        boot_indices = np.concatenate([np.arange(idx, idx + block_size) for idx in start_indices])[:T]

        r1_boot = r1_circ[boot_indices]
        r2_boot = r2_circ[boot_indices]

        sr1_b = safe_sharpe_ratio(r1_boot)
        sr2_b = safe_sharpe_ratio(r2_boot)
        if not np.isfinite(sr1_b) or not np.isfinite(sr2_b):
            continue
        diff_b = sr1_b - sr2_b

        # LW2008 block-based natural estimator for bootstrap covariance
        W_boot = np.column_stack((r1_boot, r2_boot, r1_boot**2, r2_boot**2))
        W_boot_centered = W_boot - np.mean(W_boot, axis=0)

        blocks = W_boot_centered[: l * block_size].reshape(l, block_size, 4)
        zeta = np.sum(blocks, axis=1) / np.sqrt(block_size)

        cov_boot = np.dot(zeta.T, zeta) / (l * T)
        se_b = calculate_se_from_cov(cov_boot, r1_boot, r2_boot)
        if not np.isfinite(se_b) or se_b <= 0:
            continue

        d_stars[i] = abs((diff_b - diff_orig) / se_b)

    valid = d_stars[np.isfinite(d_stars)]
    if len(valid) == 0:
        raise ValueError("All bootstrap draws failed due to invalid standard errors.")

    # 3. p-value per LW2008 Eq. (9) using valid draws
    p_value = (np.sum(valid >= d_orig) + 1) / (len(valid) + 1)

    return {
        "p_value": p_value,
        "sr_diff": diff_orig,
        "d_stat": d_orig,
        "block_size": block_size,
        "bootstrap_requested": B,
        "bootstrap_valid": len(valid),
        "sr_strategy": sr1,
        "sr_baseline": sr2,
    }


def run_ledoit_wolf_test(
    strategy_csv="Residual NW GPT 3.5.csv",
    baseline_csv= "LLM-S+finbert+analyst returns 3.5.csv", #"portfolio_returns_ew 10 years.csv",
    B=4999,
    block_size=None,
    random_state=42,
):
    df_strategy = pd.read_csv(strategy_csv)
    df_baseline = pd.read_csv(baseline_csv)

    r_strategy = df_strategy.iloc[:, 0].values
    r_baseline = df_baseline.iloc[:, 0].values

    if len(r_strategy) != len(r_baseline):
        raise ValueError(
            f"Row count mismatch after cleaning: strategy={len(r_strategy)}, baseline={len(r_baseline)}"
        )

    result = ledoit_wolf_bootstrap(
        r_strategy,
        r_baseline,
        B=B,
        block_size=block_size,
        random_state=random_state,
    )

    print("=== Ledoit-Wolf (2008) Bootstrap Test ===")
    print(f"Input rows used:          {len(r_strategy)}")
    print(f"Block size used:          {result['block_size']}")
    print(f"Bootstrap iterations:     {result['bootstrap_requested']}")
    print(f"Valid bootstrap draws:    {result['bootstrap_valid']}")
    print("-" * 44)
    print(f"Strategy Sharpe:          {result['sr_strategy']:.6f}")
    print(f"Baseline Sharpe:          {result['sr_baseline']:.6f}")
    print(f"Difference (Delta SR):    {result['sr_diff']:.6f}")
    print(f"Studentized stat (|d|):   {result['d_stat']:.6f}")
    print(f"P-value:                  {result['p_value']:.6f}")

    alpha = 0.05
    verdict = "Reject H0 (Sharpe ratios differ)" if result["p_value"] < alpha else "Fail to reject H0"
    print(f"Decision @ alpha={alpha}:   {verdict}")

    return result


if __name__ == "__main__":
    _ = run_ledoit_wolf_test()

=== Ledoit-Wolf (2008) Bootstrap Test ===
Input rows used:          31
Block size used:          4
Bootstrap iterations:     4999
Valid bootstrap draws:    4999
--------------------------------------------
Strategy Sharpe:          0.321225
Baseline Sharpe:          0.032771
Difference (Delta SR):    0.288454
Studentized stat (|d|):   2.542572
P-value:                  0.042800
Decision @ alpha=0.05:   Reject H0 (Sharpe ratios differ)


ChatGPT 4o:
p-value for NLS > EW: 0.247400

p-value for NLS > no quant: 0.284400

p-value for NLS > VW: 0.302200

ChatGPT 3.5:
p-value for Agentic AI > EW: 0.067000

p-value for Agentic AI > no quant: 0.062000

p-value for Agentic AI > VW: 0.1052

p-value for Agentic AI > LLM-S: 0.310600

p-value for Agentic AI > quant: 0.910600

p-value for Agentic AI > logistic: 0.735000

p-value for Agentic AI > analyst: 0.084400

p-value for Agentic AI > finbert: 0.487600

p-value for Agentic AI > LLM-S+analyst: 0.300600

p-value for Agentic AI > finbert+analyst: 0.030000

p-value for Agentic AI > LLM-S+finbert+analyst: 0.042800